In [ ]:
import pandas as pd

df = pd.read_csv(r'../dataset_with_labels.csv')
df.head()

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

#a = df[df["name"] == "APT."]
X = df.drop(columns=["spotify_id", "name", "artists", "snapshot_date", "country", "album_name", "album_release_date", "popularity"], axis=1, inplace=False)
y = df["popularity"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.columns)

scaler = MinMaxScaler()
encode = OneHotEncoder()


In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.metrics import mean_squared_error


preprocessing = make_column_transformer((encode, ['average_song']), (scaler, ['key', 'daily_rank', 'daily_movement', 'weekly_movement',
       'is_explicit', 'duration_ms', 'danceability', 'energy', 'key',
       'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
       'liveness', 'valence', 'tempo', 'time_signature'] ), remainder='passthrough')

pipeline = make_pipeline(
    preprocessing,
    RandomForestRegressor()
)
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)


print("Random Forests Regression")
mean_squared_error(y_test, y_pred)

In [ ]:
param_distributions = {
    'randomforestregressor__n_estimators': [50, 100, 200, 300],
    'randomforestregressor__max_depth': [5, 10, 20, 30],
}

from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import KFold

RFRGrid = RandomizedSearchCV(
    pipeline,
    param_distributions=param_distributions,
    scoring="neg_mean_squared_error",
    cv=KFold(n_splits=3, shuffle=True, random_state=42),
    n_iter=5,
    random_state=42,
    verbose=1
)

RFRGrid.fit(X_train, y_train)
print("Random Forests Regression with Grid Search")
print(RFRGrid.best_params_)
print(RFRGrid.best_score_)